# Week 1 | SciFact 搜索实验
从上往下逐格运行。你会先看到数据和正确答案，再运行手写 BM25，最后得到 300 条官方测试查询的指标。

**第 1–5 步是今晚的演示版本。**本 notebook 只用 Python 标准库，第 6 步的可选对照需要安装 `rank-bm25`。在 Colab 可直接上传文件，使用默认 CPU，按 Shift+Enter 逐格运行。

这里实现的是早期原型。分词使用英文小写与字母数字切词，BM25 使用正值 IDF、固定 `k1=1.2` 和 `b=0.75`。完整项目还要求停用词、词干化、开发集调参和混合检索等工作。

数据来源：[BEIR SciFact](https://github.com/beir-cellar/beir/wiki/Datasets-available)。

## 1. 下载并读懂三个文件
`corpus.jsonl` 是要搜索的文档，`queries.jsonl` 是查询文本，`qrels/*.tsv` 是人工相关性判断。下一格会显示一条训练查询和它的相关文档。qrels 仅用于查看答案和评分。

In [1]:
import csv
import heapq
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from io import TextIOWrapper
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

dataset_url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
archive = Path("scifact.zip")
if not archive.exists():
    print("Downloading SciFact…")
    partial = archive.with_suffix(".zip.part")
    with urlopen(dataset_url, timeout=30) as response:
        partial.write_bytes(response.read())
    partial.replace(archive)

with ZipFile(archive) as dataset:
    def read_jsonl(name):
        with dataset.open(f"scifact/{name}.jsonl") as stream:
            return {row["_id"]: row for row in map(json.loads, stream)} # stream每次读出一行，json.loads(line)把这一行JSON变成Python字典，_id做外层字典的键

    def read_qrels(split):
        judgments = defaultdict(dict)
        with dataset.open(f"scifact/qrels/{split}.tsv") as stream:
            rows = csv.reader(TextIOWrapper(stream, encoding="utf-8"), delimiter="	")
            next(rows)
            for query_id, doc_id, relevance in rows:
                judgments[query_id][doc_id] = int(relevance)
        return dict(judgments)

    corpus = read_jsonl("corpus") # 所有可搜索文档的标题和正文 -> 建立索引，展示结果
    queries = read_jsonl("queries") # 查询ID对应的查询文本 -> 给搜索函数提供输入
    train_qrels = read_qrels("train")
    test_qrels = read_qrels("test") # train_qrels / test_qrels 表示查询与哪篇文档相关，以及相关程度 -> 展示答案，计算指标

assert train_qrels.keys().isdisjoint(test_qrels)
assert all(qid in queries and all(doc in corpus for doc in qrels)
           for split in (train_qrels, test_qrels) for qid, qrels in split.items())
example_id = "9"  # A training query chosen for the walkthrough.
print(f"Documents: {len(corpus):,} | Train queries: {len(train_qrels):,} | Test queries: {len(test_qrels):,}")
print("Example query:", queries[example_id]["text"])
for doc_id, grade in train_qrels[example_id].items():
    print(f"  Judged document {doc_id} | relevance={grade} | {corpus[doc_id]['title']}")

Documents: 5,183 | Train queries: 809 | Test queries: 300
Example query: 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
  Judged document 44265107 | relevance=1 | Liver transplantation and opioid dependence.


## 2. 手写倒排索引与 BM25
`postings["fraction"]` 存每篇包含该词的文档 ID 和出现次数。搜索时只遍历查询词对应的文档。BM25 按词频、词的稀有程度和文档长度计算分数。你可以先看 `search()` 的外层循环，再看公式。

In [2]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower()) # 转成小写

def document_text(doc):
    return f"{doc.get('title', '')} {doc.get('text', '')}" # 把标题和正文拼在一起

postings = defaultdict(dict)
doc_lengths = {}

index_start = time.perf_counter()
for doc_id, doc in corpus.items():
    tokens = tokenize(document_text(doc)) # 把标题和正文拼接在一起然后转成小写
    doc_lengths[doc_id] = len(tokens)
    for term, count in Counter(tokens).items():
        postings[term][doc_id] = count # 这个词在文档中出现了几次

avg_doc_length = sum(doc_lengths.values()) / len(doc_lengths)
idf = {
    term: math.log1p((len(corpus) - len(docs) + 0.5) / (len(docs) + 0.5))
    for term, docs in postings.items()
}
print(f"Vocabulary: {len(postings):,} | Index build: {time.perf_counter() - index_start:.2f} s")

def search(query, top_k=100, k1=1.2, b=0.75): # top k 决定返回几篇文档，'k1'决定同一个词多出现几次能加多少分（但也会缩放length_factor），b决定文档的长度对分数的影响
    scores = defaultdict(float) # 每个文档的初始得分 0.0
    for term, query_count in Counter(tokenize(query)).items(): # 注意括号里的是query，所以外层循环走的是"查询"里的词
        for doc_id, term_frequency in postings.get(term, {}).items(): # 括号里的是term，term出现在上一个循环中，是查询中的一个词
            length_factor = 1 - b + b * doc_lengths[doc_id] / avg_doc_length
            term_score = term_frequency * (k1 + 1) / (term_frequency + k1 * length_factor)
            scores[doc_id] += query_count * idf[term] * term_score # idf[term]表示这个词在整个语料库中有多罕见

    # 找出前top k的文档，分数相同则按ID排。score.items()里的每个pair是（文档ID, 分数），而后面排序则会变成（-分数，文档ID）
    return heapq.nsmallest(top_k, scores.items(), key=lambda pair: (-pair[1], pair[0]))

Vocabulary: 35,734 | Index build: 1.47 s


## 3. 先演示一条训练查询
每行显示名次、BM25 分数和文章标题。带 `✓` 的结果在 qrels 中被标为相关。你可以改 `example_id` 为另一个训练查询 ID，再运行这一格。

In [3]:
query = queries[example_id]["text"]
print(query)
# enumerate 会把每一个遍历的元素包装成（索引，元素）的元组（Tuple），start = 1让返回的值从1开始
for rank, (doc_id, score) in enumerate(search(query, top_k=10), start=1):
  # train_qrels->双层嵌套字典，train_qrels[example_id]->返回文档ID和相关度，例如： "44265107": 1，这个就是train_qrels的子字典
    marker = "✓" if train_qrels[example_id].get(doc_id, 0) > 0 else " "
    print(f"{rank:>2}. {marker} {score:7.2f}  {corpus[doc_id]['title']}")

32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
 1. ✓   42.16  Liver transplantation and opioid dependence.
 2.     18.30  Maternal and perinatal outcome in severe pregnancy-related liver disease.
 3.     16.94  Linkage with methadone treatment upon release from incarceration: a promising opportunity.
 4.     16.50  Noninvasive ventilation for treatment of acute respiratory failure in patients undergoing solid organ transplantation: a randomized trial.
 5.     13.90  The missing cases of tuberculosis in Malawi: the contribution from cross-border registrations.
 6.     13.86  Reversal of hyperglycemia in mice by using human expandable insulin-producing cells differentiated from fetal liver progenitor cells.
 7.     13.72  Relationship between drug treatment services, retention, and outcomes.
 8.     12.49  Interferon alpha for the treatment of chronic hepatitis C in patients infected with human immunodeficiency virus. Hepatitis-HIV Spa

## 4. 学会用 Recall、MRR、NDCG 评价搜索
设有两篇相关文档 A 和 B，系统只在前三位找到了 B，且 B 排第 2。

- Recall@3 = 1/2，找回了一半。
- MRR@3 对这条查询的贡献为 1/2，第一个相关文档在第 2 位。
- NDCG@3 会考虑名次和相关程度。分母是把所有已标注的相关文档按正确顺序排到前面时能得到的分数。

In [10]:
# 前三名
# A、B、X   A、X、B   X、A、B   A、X、Y
''' Recall 问的是「找全了吗？」
看最后一个。已知有两篇相关文档，前三名只找到了 A，所以 Recall@3 = 1 ÷ 2 = 0.5。
前三个都找到了 A 和 B，所以即使顺序不同，Recall 都是 1。'''
def recall_at_k(ranked_ids, judgments, k):
    relevant = {doc for doc, grade in judgments.items() if grade > 0}
    return len(set(ranked_ids[:k]) & relevant) / len(relevant) if relevant else 0.0

'''MRR 问的是「用户翻到第几个，才看到第一篇有用的？」
看第三个。用户先看到 X，到第 2 位才看到 A，所以这条查询的 RR（倒数排名）是 1 ÷ 2 = 0.5。
最后一个虽然漏掉了 B，但 A 排第一，RR 还是 1。Notebook 把 300 条测试查询的 RR 取平均，就得到 MRR。
'''
def reciprocal_rank_at_k(ranked_ids, judgments, k):
    for rank, doc_id in enumerate(ranked_ids[:k], start=1):
        if judgments.get(doc_id, 0) > 0:
            return 1 / rank
    return 0.0

'''NDCG 问的是「把整个结果列表看一遍，相关文档排得够靠前吗？」
比较前两个。两种结果都找全了，第一篇相关文档也都排第一，所以 Recall 和 RR 一模一样。
但第一个的 B 排第 2，第二行的 B 排第 3。NDCG 能看出这个差别。
'''
def dcg_at_k(grades, k):
    return sum((2 ** grade - 1) / math.log2(rank + 1)
               for rank, grade in enumerate(grades[:k], start=1))

def ndcg_at_k(ranked_ids, judgments, k):
    actual = dcg_at_k([judgments.get(doc, 0) for doc in ranked_ids[:k]], k)
    ideal = dcg_at_k(sorted(judgments.values(), reverse=True), k)
    return actual / ideal if ideal else 0.0 # NDCG = 实际分 ÷ 理想分


In [9]:
judgments = {"A": 1, "B": 1}
ranked_ids = ["X", "B", "Y"]
assert recall_at_k(ranked_ids, judgments, 3) == 0.5
assert reciprocal_rank_at_k(ranked_ids, judgments, 3) == 0.5
assert math.isclose(ndcg_at_k(ranked_ids, judgments, 3), 1 / (math.log2(3) + 1))
assert ndcg_at_k(["A", "B"], {"A": 3, "B": 1}, 2) == 1.0
assert ndcg_at_k([], {}, 10) == 0.0
print("Metric sanity checks passed.")

Metric sanity checks passed.


## 5. 在官方 test split 上评估
每条查询先算一个分数，再对所有查询取平均。使用固定的 BM25 参数。之后如果要调参，用 train qrels 做开发评估。

延迟是当前机器上顺序处理单条查询的时间，包括分词、打分和取 top-100。先暖机，索引构建时间另外打印。p99 代表大约 99% 的查询耗时不超过这个值。

In [5]:
def percentile(values, percent):
    ordered = sorted(values)
    return ordered[math.ceil(percent / 100 * len(ordered)) - 1]

def evaluate(retrieve, qrels):
    ndcg_by_query = {}
    recalls, mrrs, latency_ms = [], [], []
    for qid in list(qrels)[:10]:
        retrieve(queries[qid]["text"])
    for qid, judgments in qrels.items():
        start = time.perf_counter()
        ranked_ids = retrieve(queries[qid]["text"])
        latency_ms.append((time.perf_counter() - start) * 1000)
        ndcg_by_query[qid] = ndcg_at_k(ranked_ids, judgments, 10)
        recalls.append(recall_at_k(ranked_ids, judgments, 100))
        mrrs.append(reciprocal_rank_at_k(ranked_ids, judgments, 10))

    print(f"Test queries: {len(qrels)}")
    print(f"NDCG@10: {sum(ndcg_by_query.values()) / len(qrels):.4f}")
    print(f"Recall@100: {sum(recalls) / len(qrels):.4f}")
    print(f"MRR@10: {sum(mrrs) / len(qrels):.4f}")
    print("Latency (ms): " + " | ".join(f"p{p}={percentile(latency_ms, p):.2f}" for p in (50, 95, 99)))
    return ndcg_by_query

def retrieve_bm25(query):
    return [doc_id for doc_id, score in search(query)]

bm25_ndcg = evaluate(retrieve_bm25, test_qrels)

Test queries: 300
NDCG@10: 0.6622
Recall@100: 0.8826
MRR@10: 0.6302
Latency (ms): p50=10.48 | p95=17.81 | p99=21.05


retrieve 是传进来的函数。传 retrieve_bm25，它就调用我们的 BM25。之后传 retrieve_reference，同一套代码就能评估参考库。两个函数都交回排好序的文档 ID，这让它们可以共用同一个评分流程。`evaluate()` 每条查询算一次指标，再对查询取平均，同时保留每条查询的 NDCG 供下一节比较。

计时只包住 retrieve(...) 这一段，包含分词、打分和选 top-100，单位从秒乘以 1000 变成毫秒。索引构建时间在前面单独统计。代码先跑 10 条查询预热，再正式测量。按这里的百分位算法, p50就是查询到第150条的用时，p95就是查询第285条的用时，以此类推。

所以上次跑出的 Recall@100 = 0.8826，可以这样读：对每条测试查询，计算前 100 个结果找回了多少比例的已知相关文档，再取平均，得到约 88.3%。

## 6. 有时间再做库对照
先运行下面的安装格，再运行对照。两个实现使用相同的词切分、相同的查询和 `k1, b`。这里的手写正值 IDF 与 `rank_bm25.BM25Okapi` 的 IDF 公式有差异，分数可能不同。两者使用相同的匹配文档候选规则和 tie-break。配对 bootstrap 对同一批 query 的 NDCG 差值重采样，估计平均差值的 95% 区间。

In [6]:
%pip install -q rank-bm25==0.2.2

In [7]:
from rank_bm25 import BM25Okapi

reference_doc_ids = sorted(corpus)
reference = BM25Okapi([tokenize(document_text(corpus[doc_id])) for doc_id in reference_doc_ids],
                      k1=1.2, b=0.75)

def retrieve_reference(query):
    tokens = tokenize(query)
    scores = reference.get_scores(tokens)
    candidates = set().union(*(postings.get(token, {}) for token in tokens))
    best = heapq.nsmallest(
        100,
        ((doc_id, score) for doc_id, score in zip(reference_doc_ids, scores)
         if doc_id in candidates),
        key=lambda pair: (-pair[1], pair[0]),
    )
    return [doc_id for doc_id, _ in best]

reference_ndcg = evaluate(retrieve_reference, test_qrels)
deltas = [bm25_ndcg[qid] - reference_ndcg[qid] for qid in test_qrels]
rng = random.Random(42)
bootstrap = [sum(rng.choices(deltas, k=len(deltas))) / len(deltas) for _ in range(2000)]
lower, upper = percentile(bootstrap, 2.5), percentile(bootstrap, 97.5)
print(f"Δ NDCG@10 (handwritten − reference): {sum(deltas) / len(deltas):+.4f}")
print(f"Paired bootstrap 95% CI: [{lower:+.4f}, {upper:+.4f}]")

Test queries: 300
NDCG@10: 0.6549
Recall@100: 0.8764
MRR@10: 0.6212
Latency (ms): p50=30.64 | p95=72.58 | p99=121.76
Δ NDCG@10 (handwritten − reference): +0.0074
Paired bootstrap 95% CI: [-0.0017, +0.0166]


### 手写 BM25 与参考实现对比

在 SciFact 的 300 条测试查询上，我将手写 BM25 与 `rank_bm25` 的参考实现进行了比较。两者使用相同的文档集合、分词方式和 BM25 参数（`k1=1.2`，`b=0.75`），并根据同一份人工相关性标注评估结果。

| 指标 | 手写 BM25 | `rank_bm25` |
|---|---:|---:|
| NDCG@10 | 0.6622 | 0.6549 |
| Recall@100 | 0.8826 | 0.8764 |
| MRR@10 | 0.6302 | 0.6212 |
| p50 查询延迟 | 10.48 ms | 30.64 ms |
| p95 查询延迟 | 17.81 ms | 72.58 ms |
| p99 查询延迟 | 21.05 ms | 121.76 ms |

手写版的三个检索质量指标都略高。其中，NDCG@10 的平均差值为 +0.0074，配对 bootstrap 得到的 95% 置信区间为 [−0.0017, +0.0166]。区间包含 0，因此这次实验尚不足以判断手写版的排序质量更好。两版的 IDF 计算方式有所不同，因此即使使用相同的 `k1` 和 `b`，也可能得到不同的排序。

在本次测试中，手写版的查询延迟更低，p99 从参考版的 121.76 ms 降至 21.05 ms。一个可能的原因是，手写版通过倒排索引只给包含查询词的文档打分，而参考版先对整个文档集合计算分数，再筛选候选文档。